# — captioning demo and meme overlay

The presentation-facing half: train an adapter, caption the sample images, and burn the
captions onto them.

This notebook used to carry its own training loop, a single-example one that differed
from `notebook.ipynb`'s batched version despite the two claiming to produce the same
checkpoints. Both now call the package's `train()`, so there is exactly one training
implementation in this repo.

In [1]:
import tempfile
import textwrap
from pathlib import Path

import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFont

from captioning import generate_caption
from config import Config
from training import train

# Paths are relative to notebooks/, which is the kernel's working directory.
config = Config(checkpoint_dir=Path("../checkpoints"), data_dir=Path("../data"))

## Train

The same `train()` that `make train` runs, on a reduced sample so the notebook finishes
quickly. It writes to a scratch directory rather than over the committed checkpoints.

In [2]:
scratch = Path(tempfile.gettempdir()) / "ai-residency-multimodal-captioning-final"
best_loss = train(
    Config(
        checkpoint_dir=scratch,
        data_dir=Path("../data"),
        num_samples=40,
        num_epochs=2,
        wandb_project=None,
        hf_repo_id=None,
    )
)
print(f"Best held-out loss: {best_loss:.4f}")

14:48:13 INFO     ai_residency_common.config | Seeded RNGs with 42


14:48:13 INFO     training | Training on cpu


14:48:13 INFO     httpx | HTTP Request: HEAD https://huggingface.co/openai/clip-vit-base-patch32/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"


14:48:13 INFO     httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/openai/clip-vit-base-patch32/3d74acf9a28c67741b2f4f2ea7635f0aaf6f0268/config.json "HTTP/1.1 200 OK"


14:48:13 INFO     httpx | HTTP Request: HEAD https://huggingface.co/openai/clip-vit-base-patch32/resolve/main/model.safetensors "HTTP/1.1 404 Not Found"


...

epoch 2/2: 100%|██████████| 4/4 [00:03<00:00,  1.16it/s]

14:48:26 INFO     training | Epoch 2/2  train_loss=5.0724  val_loss=5.8407


14:48:26 INFO     ai_residency_common.checkpoints | Saved checkpoint to /var/folders/rz/2164hm0x2cv9dcqz41x_4l640000gn/T/ai-residency-multimodal-captioning-final/adapter_flickr_batched.pth


14:48:26 INFO     training | Best held-out loss 5.8407 against baseline 7.4877 (beats baseline)


Best held-out loss: 5.8407


## Caption the sample images

Using the best committed checkpoint rather than the scratch one above, since it saw more
data.

In [3]:
items = ["cat", "dog", "bird", "car", "house"]
captions = {}
for item in items:
    image = Image.open(config.image_dir / f"{item}.jpg")
    captions[item] = generate_caption(image, config, max_new_tokens=15)
    print(f"{item}.jpg -> {captions[item]!r}")

14:48:26 INFO     httpx | HTTP Request: HEAD https://huggingface.co/openai/clip-vit-base-patch32/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"


14:48:26 INFO     httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/openai/clip-vit-base-patch32/3d74acf9a28c67741b2f4f2ea7635f0aaf6f0268/config.json "HTTP/1.1 200 OK"


14:48:26 INFO     httpx | HTTP Request: HEAD https://huggingface.co/openai/clip-vit-base-patch32/resolve/main/model.safetensors "HTTP/1.1 404 Not Found"


14:48:27 INFO     httpx | HTTP Request: HEAD https://huggingface.co/openai/clip-vit-base-patch32/resolve/main/model.safetensors.index.json "HTTP/1.1 404 Not Found"


14:48:27 INFO     httpx | HTTP Request: HEAD https://huggingface.co/openai/clip-vit-base-patch32/resolve/main/pytorch_model.bin "HTTP/1.1 302 Found"


...

cat.jpg -> 'The 2019-2020 season of the'
14:48:30 INFO     ai_residency_common.checkpoints | Loaded checkpoint from ../checkpoints/adapter_flickr_batched.pth


dog.jpg -> 'The 2019-2020 season of the National'
14:48:30 INFO     ai_residency_common.checkpoints | Loaded checkpoint from ../checkpoints/adapter_flickr_batched.pth


bird.jpg -> 'The 2016-17 season of the 20'
14:48:31 INFO     ai_residency_common.checkpoints | Loaded checkpoint from ../checkpoints/adapter_flickr_batched.pth


car.jpg -> 'The 2019-2020 season of the National'
14:48:31 INFO     ai_residency_common.checkpoints | Loaded checkpoint from ../checkpoints/adapter_flickr_batched.pth


house.jpg -> '1999-2000 season. The team was'


## Meme overlay

Presentation only, no model involved: white text with a black outline across the bottom
of each image.

In [4]:
def draw_meme(image_path: Path, caption: str) -> Image.Image:
    """Return the image with the caption burnt across its bottom edge."""
    image = Image.open(image_path).convert("RGB").resize((800, 600), Image.Resampling.LANCZOS)
    draw = ImageDraw.Draw(image)
    font = ImageFont.load_default(size=36)

    lines = textwrap.wrap(caption, width=34)
    y = 600 - len(lines) * 44 - 20
    for line in lines:
        width = draw.textbbox((0, 0), line, font=font)[2]
        x = (800 - width) // 2
        draw.text((x, y), line, font=font, fill="white", stroke_width=3, stroke_fill="black")
        y += 44
    return image


fig, axes = plt.subplots(1, len(items), figsize=(4 * len(items), 3.5))
for axis, item in zip(axes, items, strict=True):
    axis.imshow(draw_meme(config.image_dir / f"{item}.jpg", captions[item]))
    axis.axis("off")
plt.tight_layout()
plt.show()